# Modelo de Tarificación Dinámica — Telecomunicaciones
**Sector:** Telecomunicaciones (operador estilo Movistar/Vodafone España)  
**Objetivo:** Predecir la tarifa óptima de contratación para nuevos contratos de datos móviles  
**Dataset:** ~95.000 contratos con 5 tablas relacionadas (consumo, red, competencia, cliente)  
**Variable objetivo:** `tarifa_contratada` (€/mes) — problema de regresión  

---

## 0. Instalación de dependencias

In [ ]:
import sys
!{sys.executable} -m pip install xgboost optuna shap mlflow scikit-learn matplotlib seaborn category_encoders anthropic --quiet

## 1. Carga de datos desde SQLite — JOIN de 5 tablas

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

conn = sqlite3.connect('datos_teleco/teleco_pricing.db')

query = """
SELECT
    -- Identificadores
    con.contrato_id,
    con.cliente_id,

    -- Variables del cliente
    c.edad,
    c.segmento,
    c.comunidad_autonoma,
    c.canal_adquisicion,
    c.ingreso_estimado,
    c.antiguedad_meses,
    c.num_lineas_activas,
    c.tiene_fibra,
    c.tiene_tv,
    c.nps_score,

    -- Variables del contrato
    con.plan_contratado,
    con.datos_contratados_gb,
    con.descuento_aplicado,
    con.permanencia_meses,
    con.roaming_activo,
    con.seguro_dispositivo,

    -- Features agregadas de consumo (12 meses)
    ROUND(AVG(cm.consumo_datos_gb), 2)      AS consumo_medio_gb,
    ROUND(MAX(cm.consumo_datos_gb), 2)      AS consumo_max_gb,
    ROUND(AVG(cm.minutos_llamadas), 1)      AS minutos_medios,
    ROUND(AVG(cm.roaming_gb), 3)            AS roaming_medio_gb,
    ROUND(SUM(cm.cargo_exceso_eur), 2)      AS total_excesos_eur,
    ROUND(AVG(cm.exceso_datos_gb), 3)       AS exceso_medio_gb,

    -- Variables de red y coste
    r.tipo_red_principal,
    r.cobertura_pct,
    r.coste_red_mensual_eur,
    r.coste_interconexion_eur,
    r.margen_bruto_pct,
    r.congestion_red_pct,
    r.latencia_media_ms,

    -- Variables de mercado y competencia
    m.precio_competidor_min_eur,
    m.precio_competidor_max_eur,
    m.precio_medio_mercado_eur,
    m.elasticidad_precio_segmento,
    m.indice_penetracion_mercado,
    m.cuota_mercado_operador_pct,

    -- TARGET
    con.tarifa_contratada

FROM contratos con
JOIN clientes c ON con.cliente_id = c.cliente_id
JOIN consumo_mensual cm ON con.contrato_id = cm.contrato_id
JOIN red_costes r ON con.contrato_id = r.contrato_id
JOIN competencia_mercado m ON con.cliente_id = m.cliente_id
GROUP BY con.contrato_id
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Tarifa media: {df["tarifa_contratada"].mean():.2f}€')
print(f'Rango: {df["tarifa_contratada"].min():.2f}€ — {df["tarifa_contratada"].max():.2f}€')
df.head(3)

## 2. Análisis Exploratorio (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('EDA — Análisis de Tarificación por Variables Clave', fontsize=15, fontweight='bold')

# 1. Distribución de tarifa por segmento
for seg, color in zip(['Residencial', 'PYME', 'Corporativo'], ['#3498db', '#e74c3c', '#2ecc71']):
    df[df['segmento'] == seg]['tarifa_contratada'].plot(
        kind='hist', ax=axes[0,0], alpha=0.6, bins=40, label=seg, color=color)
axes[0,0].set_title('Distribución Tarifa por Segmento')
axes[0,0].set_xlabel('Tarifa (€/mes)')
axes[0,0].legend()

# 2. Tarifa media por plan
tarifa_plan = df.groupby('plan_contratado')['tarifa_contratada'].mean().sort_values()
tarifa_plan.plot(kind='barh', ax=axes[0,1], color='#9b59b6', edgecolor='white')
axes[0,1].set_title('Tarifa Media por Plan')
axes[0,1].set_xlabel('Tarifa media (€/mes)')

# 3. Tarifa vs consumo medio
sample = df.sample(3000, random_state=42)
colors = {'Residencial': '#3498db', 'PYME': '#e74c3c', 'Corporativo': '#2ecc71'}
for seg in ['Residencial', 'PYME', 'Corporativo']:
    mask = sample['segmento'] == seg
    axes[0,2].scatter(sample[mask]['consumo_medio_gb'],
                      sample[mask]['tarifa_contratada'],
                      alpha=0.4, s=10, label=seg, color=colors[seg])
axes[0,2].set_title('Tarifa vs Consumo Medio')
axes[0,2].set_xlabel('Consumo medio (GB)')
axes[0,2].set_ylabel('Tarifa (€/mes)')
axes[0,2].legend()

# 4. Elasticidad precio-demanda por segmento
elast = df.groupby('segmento')['elasticidad_precio_segmento'].mean().sort_values()
bars = axes[1,0].bar(elast.index, elast.values,
                      color=['#e74c3c', '#f39c12', '#3498db'], edgecolor='white')
axes[1,0].set_title('Elasticidad Precio-Demanda por Segmento')
axes[1,0].set_ylabel('Elasticidad (negativa = sensible al precio)')
for bar, val in zip(bars, elast.values):
    axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.05,
                   f'{val:.2f}', ha='center', color='white', fontweight='bold')

# 5. Margen bruto vs tarifa contratada
axes[1,1].scatter(df.sample(3000, random_state=42)['margen_bruto_pct'],
                  df.sample(3000, random_state=42)['tarifa_contratada'],
                  alpha=0.3, s=8, color='#e74c3c')
axes[1,1].set_title('Margen Bruto vs Tarifa Contratada')
axes[1,1].set_xlabel('Margen bruto (%)')
axes[1,1].set_ylabel('Tarifa (€/mes)')

# 6. Tarifa media por canal de adquisición
tarifa_canal = df.groupby('canal_adquisicion')['tarifa_contratada'].mean().sort_values(ascending=False)
tarifa_canal.plot(kind='bar', ax=axes[1,2], color='#1abc9c', edgecolor='white')
axes[1,2].set_title('Tarifa Media por Canal de Adquisición')
axes[1,2].tick_params(axis='x', rotation=20)
axes[1,2].set_ylabel('Tarifa media (€/mes)')

plt.tight_layout()
plt.savefig('eda_tarificacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Gráfico guardado: eda_tarificacion.png')

In [ ]:
# Análisis de elasticidad precio-demanda por segmento (log-log)
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Análisis de Elasticidad Precio-Demanda (Regresión Log-Log)', fontsize=13, fontweight='bold')

resultados_elasticidad = {}

for i, seg in enumerate(['Residencial', 'PYME', 'Corporativo']):
    datos_seg = df[df['segmento'] == seg].copy()
    datos_seg = datos_seg[datos_seg['tarifa_contratada'] > 0]
    datos_seg = datos_seg[datos_seg['consumo_medio_gb'] > 0]

    log_precio  = np.log(datos_seg['tarifa_contratada'])
    log_consumo = np.log(datos_seg['consumo_medio_gb'])

    slope, intercept, r, p, _ = stats.linregress(log_precio, log_consumo)
    resultados_elasticidad[seg] = {'elasticidad': round(slope, 3), 'R2': round(r**2, 3)}

    sample_seg = datos_seg.sample(min(2000, len(datos_seg)), random_state=42)
    axes[i].scatter(np.log(sample_seg['tarifa_contratada']),
                    np.log(sample_seg['consumo_medio_gb']),
                    alpha=0.2, s=6, color=['#3498db', '#e74c3c', '#2ecc71'][i])

    x_line = np.linspace(log_precio.min(), log_precio.max(), 100)
    axes[i].plot(x_line, intercept + slope * x_line, 'k-', linewidth=2)
    axes[i].set_title(f'{seg}\nElasticidad: {slope:.3f} | R²: {r**2:.3f}')
    axes[i].set_xlabel('Log(Tarifa €/mes)')
    axes[i].set_ylabel('Log(Consumo GB)')

plt.tight_layout()
plt.savefig('elasticidad_log_log.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nResultados de elasticidad por segmento:')
for seg, res in resultados_elasticidad.items():
    print(f'  {seg:<12}: elasticidad={res["elasticidad"]:+.3f} | R²={res["R2"]:.3f}')
print('\nInterpretación: elasticidad < 0 → a mayor precio, menor consumo (sensibilidad al precio)')

In [ ]:
# Matriz de correlación con la variable objetivo
numericas = [
    'edad', 'ingreso_estimado', 'antiguedad_meses', 'num_lineas_activas',
    'datos_contratados_gb', 'descuento_aplicado', 'consumo_medio_gb',
    'minutos_medios', 'total_excesos_eur', 'coste_red_mensual_eur',
    'margen_bruto_pct', 'precio_medio_mercado_eur', 'elasticidad_precio_segmento',
    'tarifa_contratada'
]

plt.figure(figsize=(13, 10))
corr = df[numericas].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matriz de Correlación — Variables Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlacion_teleco.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocesamiento y Feature Engineering

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder

# ── Feature engineering ──────────────────────────────────────────
df['ratio_consumo_contratado']  = (df['consumo_medio_gb'] / df['datos_contratados_gb'].replace(999, 100)).round(4)
df['coste_total_mensual']       = (df['coste_red_mensual_eur'] + df['coste_interconexion_eur']).round(2)
df['diferencial_vs_mercado']    = (df['tarifa_contratada'] - df['precio_medio_mercado_eur']).round(2)
df['indice_multiproducto']      = df['tiene_fibra'] + df['tiene_tv'] + (df['num_lineas_activas'] > 1).astype(int)
df['valor_cliente_estimado']    = (df['ingreso_estimado'] * df['antiguedad_meses'] / 1000).round(2)
df['saturacion_datos']          = (df['consumo_medio_gb'] / df['datos_contratados_gb'].replace(999, 100)).clip(0, 1.5).round(4)

# ── Definir features ─────────────────────────────────────────────
categoricas = ['segmento', 'comunidad_autonoma', 'canal_adquisicion',
               'plan_contratado', 'tipo_red_principal']

numericas_modelo = [
    'edad', 'ingreso_estimado', 'antiguedad_meses', 'num_lineas_activas',
    'tiene_fibra', 'tiene_tv', 'nps_score',
    'datos_contratados_gb', 'descuento_aplicado', 'permanencia_meses',
    'roaming_activo', 'seguro_dispositivo',
    'consumo_medio_gb', 'consumo_max_gb', 'minutos_medios',
    'roaming_medio_gb', 'total_excesos_eur', 'exceso_medio_gb',
    'cobertura_pct', 'coste_red_mensual_eur', 'coste_interconexion_eur',
    'margen_bruto_pct', 'congestion_red_pct', 'latencia_media_ms',
    'precio_competidor_min_eur', 'precio_competidor_max_eur',
    'precio_medio_mercado_eur', 'elasticidad_precio_segmento',
    'indice_penetracion_mercado', 'cuota_mercado_operador_pct',
    'ratio_consumo_contratado', 'coste_total_mensual',
    'indice_multiproducto', 'valor_cliente_estimado', 'saturacion_datos'
]

X = df[numericas_modelo + categoricas].copy()
y = df['tarifa_contratada']

# ── Split train/test ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Target Encoding para categóricas ─────────────────────────────
te = TargetEncoder(cols=categoricas)
X_train = te.fit_transform(X_train, y_train)
X_test  = te.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Tarifa media train: {y_train.mean():.2f}€ | test: {y_test.mean():.2f}€')
print(f'Features totales: {X_train.shape[1]}')

## 4. Modelado — Comparación de modelos de regresión

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score
import mlflow
import mlflow.sklearn

mlflow.set_experiment('tarificacion_dinamica_teleco')

def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)
    mae    = mean_absolute_error(y_te, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred))
    r2     = modelo.score(X_te, y_te)
    print(f'{nombre:<30} MAE: {mae:.2f}€ | RMSE: {rmse:.2f}€ | R²: {r2:.4f}')
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'y_pred': y_pred}

resultados = {}

# ── Ridge Regression (baseline regularizado) ──────────────────────
with mlflow.start_run(run_name='Ridge_baseline'):
    ridge = Ridge(alpha=10.0)
    res = evaluar_modelo('Ridge Regression', ridge, X_train, y_train, X_test, y_test)
    mlflow.log_metrics({'mae': res['mae'], 'rmse': res['rmse'], 'r2': res['r2']})
    mlflow.sklearn.log_model(ridge, 'ridge')
    resultados['Ridge'] = res

# ── Lasso Regression ──────────────────────────────────────────────
with mlflow.start_run(run_name='Lasso'):
    lasso = Lasso(alpha=0.1, max_iter=5000)
    res = evaluar_modelo('Lasso Regression', lasso, X_train, y_train, X_test, y_test)
    mlflow.log_metrics({'mae': res['mae'], 'rmse': res['rmse'], 'r2': res['r2']})
    mlflow.sklearn.log_model(lasso, 'lasso')
    resultados['Lasso'] = res

# ── Random Forest ─────────────────────────────────────────────────
with mlflow.start_run(run_name='RandomForest'):
    rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
    res = evaluar_modelo('Random Forest', rf, X_train, y_train, X_test, y_test)
    mlflow.log_metrics({'mae': res['mae'], 'rmse': res['rmse'], 'r2': res['r2']})
    mlflow.sklearn.log_model(rf, 'random_forest')
    resultados['Random Forest'] = res

print('\nBaseline completado. Pasamos a XGBoost + Optuna...')

In [ ]:
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 600),
        'max_depth'       : trial.suggest_int('max_depth', 3, 12),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state'    : 42,
        'n_jobs'          : -1
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, verbose=False)
    y_pred = model.predict(X_test)
    return mean_absolute_error(y_test, y_pred)

print('Optimizando hiperparámetros con Optuna (150 trials)...')
study = optuna.create_study(direction='minimize')  # minimizar MAE
study.optimize(objective, n_trials=150, show_progress_bar=True)

print(f'\nMejor MAE en optimización: {study.best_value:.2f}€')
print(f'Mejores hiperparámetros: {study.best_params}')

In [ ]:
# Modelo final XGBoost con validación cruzada k-fold
from sklearn.model_selection import KFold

with mlflow.start_run(run_name='XGBoost_Optuna_final'):
    best_params = study.best_params
    best_params.update({'random_state': 42, 'n_jobs': -1})

    xgb_final = xgb.XGBRegressor(**best_params)
    xgb_final.fit(X_train, y_train, verbose=False)
    y_pred = xgb_final.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = xgb_final.score(X_test, y_test)

    # Validación cruzada k-fold (k=5)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    X_all = pd.concat([X_train, X_test])
    y_all = pd.concat([y_train, y_test])
    cv_scores = cross_val_score(xgb_final, X_all, y_all,
                                cv=kf, scoring='neg_mean_absolute_error')
    mae_cv = -cv_scores.mean()

    # MAE por segmento
    test_df = X_test.copy()
    test_df['y_real'] = y_test.values
    test_df['y_pred'] = y_pred
    test_df['segmento_orig'] = df.loc[y_test.index, 'segmento'].values

    mae_segmento = {}
    for seg in ['Residencial', 'PYME', 'Corporativo']:
        mask = test_df['segmento_orig'] == seg
        mae_seg = mean_absolute_error(test_df[mask]['y_real'], test_df[mask]['y_pred'])
        mae_segmento[seg] = round(mae_seg, 2)

    mlflow.log_params(best_params)
    mlflow.log_metrics({'mae': mae, 'rmse': rmse, 'r2': r2, 'mae_cv_5fold': mae_cv})
    mlflow.xgboost.log_model(xgb_final, 'xgboost_final')
    resultados['XGBoost + Optuna'] = {'mae': mae, 'rmse': rmse, 'r2': r2, 'y_pred': y_pred}

    print('═' * 55)
    print('MÉTRICAS MODELO FINAL — XGBoost + Optuna')
    print('═' * 55)
    print(f'MAE global     : {mae:.2f}€')
    print(f'RMSE global    : {rmse:.2f}€')
    print(f'R²             : {r2:.4f}')
    print(f'MAE CV 5-fold  : {mae_cv:.2f}€')
    print(f'\nMAE por segmento:')
    for seg, mae_s in mae_segmento.items():
        print(f'  {seg:<12}: {mae_s:.2f}€')

In [ ]:
# Visualización comparativa de modelos y análisis de residuos
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Evaluación del Modelo de Tarificación', fontsize=14, fontweight='bold')

# 1. Comparativa MAE por modelo
modelos_nombres = list(resultados.keys())
maes = [resultados[m]['mae'] for m in modelos_nombres]
colores = ['#95a5a6', '#95a5a6', '#3498db', '#e74c3c']
bars = axes[0,0].bar(modelos_nombres, maes, color=colores, edgecolor='white')
axes[0,0].set_title('Comparativa MAE por Modelo (€)')
axes[0,0].set_ylabel('MAE (€)')
for bar, val in zip(bars, maes):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                   f'{val:.2f}€', ha='center', fontweight='bold', fontsize=9)
axes[0,0].tick_params(axis='x', rotation=15)

# 2. Real vs Predicho
sample_idx = np.random.choice(len(y_test), 2000, replace=False)
axes[0,1].scatter(y_test.values[sample_idx], y_pred[sample_idx],
                  alpha=0.3, s=8, color='#e74c3c')
min_val, max_val = y_test.min(), y_test.max()
axes[0,1].plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.5, label='Predicción perfecta')
axes[0,1].set_title(f'Real vs Predicho | R²={r2:.4f}')
axes[0,1].set_xlabel('Tarifa real (€/mes)')
axes[0,1].set_ylabel('Tarifa predicha (€/mes)')
axes[0,1].legend()

# 3. Distribución de residuos
residuos = y_test.values - y_pred
axes[1,0].hist(residuos, bins=60, color='#3498db', edgecolor='white', alpha=0.8)
axes[1,0].axvline(0, color='red', linewidth=2, linestyle='--')
axes[1,0].set_title(f'Distribución de Residuos | Media: {residuos.mean():.2f}€')
axes[1,0].set_xlabel('Error (€)')
axes[1,0].set_ylabel('Frecuencia')

# 4. MAE por segmento
segs = list(mae_segmento.keys())
maes_seg = list(mae_segmento.values())
axes[1,1].bar(segs, maes_seg, color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='white')
axes[1,1].set_title('MAE por Segmento de Cliente')
axes[1,1].set_ylabel('MAE (€)')
for i, (seg, mae_s) in enumerate(mae_segmento.items()):
    axes[1,1].text(i, mae_s + 0.05, f'{mae_s:.2f}€', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('evaluacion_modelo.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Explicabilidad — SHAP Values

In [ ]:
import shap

print('Calculando SHAP values...')
explainer   = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar',
                  feature_names=X_test.columns.tolist(),
                  show=False, max_display=20)
plt.title('Variables más importantes para la Tarificación (SHAP)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_tarificacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis de residuos por segmento — detectar sesgos del modelo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Análisis de Residuos por Segmento', fontsize=13, fontweight='bold')

for i, seg in enumerate(['Residencial', 'PYME', 'Corporativo']):
    mask = test_df['segmento_orig'] == seg
    res_seg = test_df[mask]['y_real'] - test_df[mask]['y_pred']
    axes[i].hist(res_seg, bins=40, edgecolor='white',
                 color=['#3498db', '#e74c3c', '#2ecc71'][i], alpha=0.8)
    axes[i].axvline(0, color='black', linewidth=2, linestyle='--')
    axes[i].axvline(res_seg.mean(), color='red', linewidth=1.5,
                    linestyle='-', label=f'Media: {res_seg.mean():.2f}€')
    axes[i].set_title(f'{seg}\nMAE: {mae_segmento[seg]:.2f}€')
    axes[i].set_xlabel('Error de predicción (€)')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig('residuos_segmento.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Módulo GenAI — Recomendación de tarifa en lenguaje natural

In [ ]:
import anthropic
import os

# Pon tu API key aquí o usa variable de entorno
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

def generar_recomendacion_tarifa(perfil_cliente: dict, tarifa_predicha: float,
                                  top_shap_features: list,
                                  contexto_mercado: dict) -> str:
    """
    Genera una recomendación comercial de tarifa en lenguaje natural
    para el equipo de ventas, combinando la predicción del modelo
    con el contexto de mercado y los drivers de precio (SHAP).
    """
    features_texto = '\n'.join([
        f'  - {feat}: valor={val_real:.2f} | impacto en tarifa={shap_val:+.2f}€'
        for feat, shap_val, val_real in top_shap_features
    ])

    prompt = f"""Eres un experto en estrategia de pricing para operadores de telecomunicaciones.
Tu rol es asesorar al equipo comercial sobre la tarifa óptima a ofrecer a un nuevo cliente,
maximizando el margen sin comprometer la conversión.

TARIFA RECOMENDADA POR EL MODELO: {tarifa_predicha:.2f}€/mes

PERFIL DEL CLIENTE:
{chr(10).join([f'  - {k}: {v}' for k, v in perfil_cliente.items()])}

FACTORES QUE MÁS DETERMINAN LA TARIFA (SHAP):
{features_texto}

CONTEXTO DE MERCADO:
  - Precio mínimo competencia: {contexto_mercado['precio_min']}€
  - Precio máximo competencia: {contexto_mercado['precio_max']}€
  - Precio medio mercado: {contexto_mercado['precio_medio']}€
  - Elasticidad del segmento: {contexto_mercado['elasticidad']}

Redacta una recomendación comercial de 3 párrafos que:
1. Justifique la tarifa recomendada en términos de negocio
2. Indique el rango de negociación aceptable y por qué
3. Señale qué argumentos usar con el cliente según su perfil
Sin encabezados. Tono profesional y directo."""

    respuesta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        messages=[{"role": "user", "content": prompt}]
    )
    return respuesta.content[0].text


# ── Ejemplo con un cliente corporativo del test set ───────────────
idx_corp = test_df[test_df['segmento_orig'] == 'Corporativo'].index[0]
pos_idx  = list(X_test.index).index(idx_corp)

shap_idx     = shap_values[pos_idx]
top_idx      = np.argsort(np.abs(shap_idx))[-5:][::-1]
top_features = [
    (X_test.columns[i], shap_idx[i], X_test.iloc[pos_idx, i])
    for i in top_idx
]

cliente_real = df.loc[idx_corp]
perfil = {
    'Segmento'            : cliente_real['segmento'],
    'Plan solicitado'     : cliente_real['plan_contratado'],
    'Datos contratados'   : f"{cliente_real['datos_contratados_gb']} GB",
    'Número de líneas'    : int(cliente_real['num_lineas_activas']),
    'Antigüedad'          : f"{cliente_real['antiguedad_meses']} meses",
    'Tiene fibra'         : 'Sí' if cliente_real['tiene_fibra'] else 'No',
    'Tiene TV'            : 'Sí' if cliente_real['tiene_tv'] else 'No',
    'Consumo medio'       : f"{cliente_real['consumo_medio_gb']:.1f} GB/mes",
    'NPS score'           : int(cliente_real['nps_score']),
}

contexto = {
    'precio_min'  : round(cliente_real['precio_competidor_min_eur'], 2),
    'precio_max'  : round(cliente_real['precio_competidor_max_eur'], 2),
    'precio_medio': round(cliente_real['precio_medio_mercado_eur'], 2),
    'elasticidad' : round(cliente_real['elasticidad_precio_segmento'], 3),
}

tarifa_pred = round(y_pred[pos_idx], 2)
print(f'Tarifa predicha por el modelo: {tarifa_pred}€/mes')
print(f'Tarifa real contratada: {cliente_real["tarifa_contratada"]}€/mes')
print('─' * 60)
recomendacion = generar_recomendacion_tarifa(perfil, tarifa_pred, top_features, contexto)
print(recomendacion)

## 7. Guardar modelo final

In [ ]:
import pickle

artefactos = {
    'modelo'         : xgb_final,
    'target_encoder' : te,
    'features'       : numericas_modelo + categoricas,
    'mae_segmento'   : mae_segmento,
    'metricas': {
        'mae'      : round(mae, 4),
        'rmse'     : round(rmse, 4),
        'r2'       : round(r2, 4),
        'mae_cv'   : round(mae_cv, 4)
    }
}

with open('modelo_tarificacion_v1.pkl', 'wb') as f:
    pickle.dump(artefactos, f)

print('✅ Modelo guardado: modelo_tarificacion_v1.pkl')
print(f'\nMétricas finales:')
for k, v in artefactos['metricas'].items():
    print(f'  {k:<10}: {v}')
print(f'\nMAE por segmento:')
for seg, mae_s in mae_segmento.items():
    print(f'  {seg:<12}: {mae_s:.2f}€')
print(f'\n🚀 Siguiente paso: FastAPI + Docker (pricing_api.py)')